# Importação das bibliotecas necessárias

In [16]:
import sys
import os
sys.path.append('/home/jovyan/work')

from utils.AlinharETL import AlinharETL

# Crie uma instância da classe AlinharETL

In [17]:
# Crie uma instância da classe
bucket = "silver"
datamart = "SGT"
extrator_bronze = AlinharETL(bucket,datamart)

2024-10-09 14:53:26,193 - INFO - Iniciando Sessão Spark.


# Leitura da tabela

In [18]:
df_atendimentos_integracoes_sgt = extrator_bronze.criar_view_temporaria('bronze/SGT/AtendimentosIntegracoesSgt','atendimentosintegracoessgt')

2024-10-09 14:53:26,945 - INFO - Aguarde... Criando View (bronze/SGT/AtendimentosIntegracoesSgt)
2024-10-09 14:53:32,427 - INFO - View criada com sucesso


In [19]:
sql_query = """
SELECT 
    idAtendimento as id_atendimento,
    dataEmissao as dt_emissao,
    dataNegociacao as dt_negociacao,
    dataAceitacao as dt_aceitacao,
    dataExecucao as dt_execucao,
    dataRecusa as dt_recusa,
    dataConclusao as dt_conclusao,
    dataCancelamento as dt_cancelamento
FROM 
    atendimentosintegracoessgt
WHERE idAtendimento IS NOT NULL;
"""

In [20]:
df_atendimentos_integracoes_sgt = extrator_bronze.carregar_dados_delta(sql_query)

2024-10-09 14:53:34,841 - INFO - Aguarde... Executando Query (Delta)
2024-10-09 14:53:34,842 - INFO - 
SELECT 
    idAtendimento as id_atendimento,
    dataEmissao as dt_emissao,
    dataNegociacao as dt_negociacao,
    dataAceitacao as dt_aceitacao,
    dataExecucao as dt_execucao,
    dataRecusa as dt_recusa,
    dataConclusao as dt_conclusao,
    dataCancelamento as dt_cancelamento
FROM 
    atendimentosintegracoessgt
WHERE idAtendimento IS NOT NULL;

2024-10-09 14:53:34,896 - INFO - Query (Delta) executada com sucesso


In [23]:
df_atendimentos_integracoes_sgt.show(truncate = False)

+--------------+----------------------+--------------------------+--------------------------+--------------------------+----------------------+--------------------------+----------------------+--------------------------+
|id_atendimento|dt_emissao            |dt_negociacao             |dt_aceitacao              |dt_execucao               |dt_recusa             |dt_conclusao              |dt_cancelamento       |dt_atualizacao            |
+--------------+----------------------+--------------------------+--------------------------+--------------------------+----------------------+--------------------------+----------------------+--------------------------+
|97267         |2018-02-28 00:00:00+00|2018-02-28 17:09:52.589+00|2018-02-28 17:41:58.939+00|2018-03-26 17:26:28.156+00|1970-01-01 00:00:00+00|2018-07-31 19:45:26.844+00|1970-01-01 00:00:00+00|2024-10-09 14:54:10.952584|
|97631         |2018-02-28 00:00:00+00|2018-02-28 19:26:25.624+00|2018-02-28 19:29:09.601+00|2018-02-28 19:30:54.866

# Gravação no datalake

In [6]:
extrator_bronze.caminho_tabela_delta = 's3a://silver/SGT/AtendimentosIntegracoesSgt'

In [7]:
extrator_bronze.salvar_delta(df_atendimentos_integracoes_sgt, 'overwrite')

2024-09-25 17:07:37,512 - INFO - Aguarde... Persistindo dados (overwrite)
2024-09-25 17:07:45,958 - INFO - Dados persistidos com sucesso
2024-09-25 17:07:45,959 - INFO - s3a://silver/SGT/AtendimentosIntegracoesSgt
2024-09-25 17:07:45,960 - INFO - Aguarde... Realizando optimize
2024-09-25 17:07:48,430 - INFO - Optimize executado com sucesso.
2024-09-25 17:07:48,431 - INFO - Aguarde... Realizando vacuum
2024-09-25 17:08:16,771 - INFO - Vacuum executado com sucesso.


# Encerra sessão spark

In [8]:
extrator_bronze.parar_sessao()

2024-09-25 17:08:17,197 - INFO - Sessão Spark finalizada.
